# Federated Skin Lesion Classifier - Exploration Notebook

This notebook demonstrates:
1. Loading HAM10000 metadata
2. Visualizing class distribution
3. Inspecting sample dermoscopic images
4. Running a quick centralized training loop

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

from src.utils.config import load_config

In [ ]:
config = load_config('../config.yaml')
metadata_path = Path(config['dataset']['processed_dir']) / 'HAM10000_metadata.csv'

if not metadata_path.exists():
    print('Metadata not found. Run python scripts/download_data.py first.')
else:
    df = pd.read_csv(metadata_path)
    display(df.head())
    print(df['label'].value_counts())

In [ ]:
# Class distribution
if metadata_path.exists():
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x='label', order=df['label'].value_counts().index)
    plt.title('HAM10000 Class Distribution')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show sample images
image_dir = Path(config['dataset']['raw_dir']) / 'images'
if image_dir.exists():
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    for idx, row in df.groupby('label').head(1).iterrows():
        img_path = image_dir / f"{row['image_id']}.jpg"
        if img_path.exists():
            axes[idx].imshow(Image.open(img_path))
            axes[idx].set_title(row['label'])
            axes[idx].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Quick centralized training
from scripts.train_centralized import train_centralized

cfg = load_config('../config.yaml')
cfg['centralized']['epochs'] = 5
metrics = train_centralized(cfg, use_synthetic=True)
print(metrics)